# Full_full LoRaX site-level MIL / attention screen

Plot-only notebook for mixed-granularity molecule-site GIN x mean protein ESM-1b experiments.
Six methods are compared: `boost_gin_mean` (XGBoost control on mean-pooled embeddings), three
permutation-invariant MIL pooling heads (`mil_max`, `mil_noisy_or`, `mil_lse`), and two learned
attention heads over molecule sites (`attention_mil`, `attention_excess_max`).

Data protocol:

- transductive: genuine LoRaX `rand_split_1..5`;
- inductive_molecule: cold-molecule split seeds `42..46` from the fold-1 full pair pool;
- molecule keys: LoRaX SMILES -> InChIKey via `data/processed/molecules/lorax_smiles_to_inchikey.csv`;
- molecule embeddings: `gin_supervised_contextpred_all_m2or[_per_atom].npz`;
- protein embeddings: LoRaX `esm1b_650m_mean_lorax.npz`.

Run script:

```bash
bash scripts/queues/run_full_full_site_mil_attention.sh
```

In [ ]:
from pathlib import Path
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import t as student_t

plt.rcParams.update({
    "figure.figsize": (11, 4),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
COLORS = {
    "transductive": "#4C78A8",
    "inductive_molecule": "#F58518",
}
MODEL_ORDER = ["boost_gin_mean", "mil_max", "mil_noisy_or", "mil_lse", "attention_mil", "attention_excess_max"]

ROOT = Path("..").resolve().parent if Path.cwd().name == "baselines_research" else Path.cwd()
RESULT_DIR = ROOT / "results" / "attention" / "full_full" / "site_mil"
METRICS = ["AUROC", "AUPRC", "MCC", "F1"]
metrics_path = RESULT_DIR / "metrics.csv"
print("results:", RESULT_DIR)
print("metrics exists:", metrics_path.exists())

In [ ]:
if metrics_path.exists():
    runs = pd.read_csv(metrics_path)
else:
    runs = pd.DataFrame(columns=["regime", "repeat", "model", *METRICS])

display_cols = ["regime", "repeat", "model", "threshold", "n_train", "n_val", "n_test", "test_pos", *METRICS]
available_cols = [c for c in display_cols if c in runs.columns]
print(f"loaded {len(runs)} runs")
if len(runs):
    display(runs[available_cols].sort_values(["regime", "model", "repeat"]).head(20))
else:
    print("No runs yet. Launch scripts/queues/run_full_full_site_mil_attention.sh on the server.")

In [ ]:
def ci95(x):
    x = np.asarray(pd.Series(x).dropna(), dtype=float)
    n = len(x)
    if n == 0:
        return pd.Series({"mean": np.nan, "sd": np.nan, "ci_low": np.nan, "ci_high": np.nan, "n": 0})
    mean = float(x.mean())
    sd = float(x.std(ddof=1)) if n > 1 else np.nan
    half = float(student_t.ppf(0.975, n - 1) * sd / math.sqrt(n)) if n > 1 else np.nan
    return pd.Series({"mean": mean, "sd": sd, "ci_low": mean - half if n > 1 else np.nan, "ci_high": mean + half if n > 1 else np.nan, "n": n})

if len(runs):
    summary = (
        runs.groupby(["regime", "model"])[METRICS]
        .agg(lambda s: ci95(s)["mean"])
        .reset_index()
        .sort_values(["regime", "AUPRC"], ascending=[True, False])
    )
    ci_rows = []
    for (regime, model), g in runs.groupby(["regime", "model"]):
        for metric in METRICS:
            row = ci95(g[metric]).to_dict()
            row.update({"regime": regime, "model": model, "metric": metric})
            ci_rows.append(row)
    ci = pd.DataFrame(ci_rows)
    display(summary)
else:
    summary = pd.DataFrame()
    ci = pd.DataFrame()

In [ ]:
# Grouped bar chart, one panel per metric: model on the x-axis, regime as bar color,
# 95% CI (Student-t over repeats) as error bars.
if len(ci):
    models_present = [m for m in MODEL_ORDER if m in ci["model"].unique()] or sorted(ci["model"].unique())
    regimes_present = [r for r in COLORS if r in ci["regime"].unique()] or sorted(ci["regime"].unique())

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    axes = axes.ravel()
    x = np.arange(len(models_present))
    width = 0.8 / max(len(regimes_present), 1)

    for ax, metric in zip(axes, METRICS):
        d = ci[ci.metric == metric]
        for i, regime in enumerate(regimes_present):
            g = d[d.regime == regime].set_index("model").reindex(models_present)
            offset = (i - (len(regimes_present) - 1) / 2) * width
            err_low = (g["mean"] - g["ci_low"]).to_numpy()
            err_high = (g["ci_high"] - g["mean"]).to_numpy()
            err = np.vstack([np.nan_to_num(err_low), np.nan_to_num(err_high)])
            bars = ax.bar(x + offset, g["mean"], width=width, yerr=err, capsize=3,
                           color=COLORS.get(regime, None), alpha=0.85, label=regime)
            ax.bar_label(bars, fmt="%.3f", fontsize=7, padding=2, rotation=90)
        ax.set_title(metric)
        ax.set_xticks(x)
        ax.set_xticklabels(models_present, rotation=30, ha="right")
        ax.set_ylim(0, 1)
        ax.set_ylabel(metric)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=len(regimes_present), frameon=False, bbox_to_anchor=(0.5, 1.04))
    fig.suptitle("Full_full LoRaX site-MIL/attention screen: mean +/- 95% CI over repeats", y=1.08)
    fig.tight_layout()
    plt.show()
else:
    print("No CI plot yet.")

In [ ]:
if len(runs):
    coverage_cols = [c for c in runs.columns if c.endswith("_rows_after") or c in ["regime", "repeat", "model", "n_train", "n_val", "n_test"]]
    coverage = runs[coverage_cols].drop_duplicates(["regime", "repeat"]).sort_values(["regime", "repeat"])
    display(coverage)
else:
    print("No coverage table yet.")

In [ ]:
# Best-by-AUPRC test score distributions: one row per model, one column per regime,
# positive/negative pairs overlaid as density histograms.
pred_dir = RESULT_DIR / "predictions"
if len(runs) and pred_dir.exists():
    best = runs.sort_values(["regime", "AUPRC"], ascending=[True, False]).groupby("regime").head(1)
    pred_frames = []
    for row in best.itertuples():
        run_id = row.run_id if hasattr(row, "run_id") else f"{row.regime}_rep{row.repeat}_{row.model}_seed{row.seed}_boost{row.boost_seed}"
        path = pred_dir / f"test_{run_id}.csv"
        if path.exists():
            df = pd.read_csv(path)
            df["regime"] = row.regime
            df["model"] = row.model
            pred_frames.append(df)

    if pred_frames:
        pred = pd.concat(pred_frames, ignore_index=True)
        row_models = [m for m in MODEL_ORDER if m in pred["model"].unique()] or sorted(pred["model"].unique())
        col_regimes = [r for r in COLORS if r in pred["regime"].unique()] or sorted(pred["regime"].unique())

        fig, axes = plt.subplots(len(row_models), len(col_regimes),
                                  figsize=(5.2 * len(col_regimes), 3.0 * len(row_models)),
                                  squeeze=False)
        for i, model in enumerate(row_models):
            for j, regime in enumerate(col_regimes):
                ax = axes[i][j]
                g = pred[(pred.model == model) & (pred.regime == regime)]
                if g.empty:
                    ax.axis("off")
                    continue
                ax.hist(g.loc[g.y == 0, "p"], bins=40, density=True, alpha=0.55,
                        color="#4C78A8", label="negative")
                ax.hist(g.loc[g.y == 1, "p"], bins=40, density=True, alpha=0.55,
                        color="#D62728", label="positive")
                if i == 0:
                    ax.set_title(regime)
                if j == 0:
                    ax.set_ylabel(model, fontsize=9)
                if i == len(row_models) - 1:
                    ax.set_xlabel("p")
                if i == 0 and j == 0:
                    ax.legend(frameon=False, fontsize=8)
        fig.suptitle("Best-by-AUPRC test score distributions", y=1.02)
        fig.tight_layout()
        plt.show()
    else:
        print("Prediction files for best runs were not found yet.")
else:
    print("No predictions yet.")